# ice9 basic tier

The basic tier runs several AI models on your image and combines their outputs. It takes longer than the free tier — up to 2 minutes — because it waits for all the models to finish.

What you get:
- A **summary caption** describing the image, synthesised from all the model outputs
- **Validated nouns** — things the models agreed are in the image, confirmed with bounding boxes
- **Individual model outputs** — what each model said before the results were combined
- Everything from the free tier (nudenet, colors, metadata, ocr, qr)

**Run the first cell once.** The rest of the notebook explores that single result — you don't need to resubmit.

**Before you start:**
- Set your API key: `export ICE9_API_KEY=ice9_...` in your terminal before launching Jupyter, or set it in the cell below.
- Install dependencies: `pip install ice9 Pillow`

In [ ]:
from ice9 import Ice9
from ice9.exceptions import AnalysisTimeoutError, PartialResultError

# Set your image path here
IMAGE = "path/to/your/image.jpg"

# If you didn't set ICE9_API_KEY in your environment, you can set it here instead:
# import os
# os.environ["ICE9_API_KEY"] = "ice9_..."

client = Ice9(timeout=120.0)  # VLMs can take a while

print("Submitting image — this may take up to 2 minutes...")
try:
    result = client.analyze(IMAGE, tier="basic")
except AnalysisTimeoutError:
    print("Timed out. The server may be under load — try again.")
    raise
except PartialResultError as e:
    print(f"Warning: some services failed: {e.result.services_failed}")
    result = e.result

print(f"Done. Image ID: {result.image_id}")
print(f"Services: {result.services_submitted}")

## Summary caption

A single description of the image, generated by synthesising all the individual model outputs.

In [ ]:
if result.caption:
    print(result.caption)
else:
    print("No caption yet.")

## Validated nouns

These are the things the models agreed are in the image. The `vote_count` shows how many models mentioned it. The ones marked `grounding_validated` were also confirmed by Florence-2, which found matching regions in the image.

In [ ]:
if result.nouns is not None:
    for noun in result.nouns.validated:
        confirmed = "✓ confirmed" if noun["grounding_validated"] else ""
        print(f"{noun['canonical']:20s}  votes={noun['vote_count']}  {confirmed}")
else:
    print("No validated nouns yet.")

You can also see all the nouns that were extracted, including ones that didn't make the confidence threshold:

In [ ]:
if result.nouns is not None:
    for noun in result.nouns.consensus:
        print(f"{noun['canonical']:20s}  votes={noun['vote_count']}  confidence={noun['confidence']:.0%}")

## Florence-2 bounding boxes

Florence-2 found these regions in the image — one per noun phrase it could locate.

In [ ]:
if result.nouns is not None and result.nouns.regions:
    for region in result.nouns.regions:
        label = region.get("label") or region.get("text") or "unknown"
        bbox = region.get("bbox") or region.get("quad_box")
        print(f"{label:25s}  {bbox}")
else:
    print("No regions available.")

## Individual model outputs

What each AI model said about the image before the results were combined.

In [ ]:
skip = ("nudenet", "colors", "metadata", "ocr", "qr",
        "florence2_grounding", "noun_consensus",
        "caption_summary", "verb_consensus", "consensus")

for service_name in result.services_submitted:
    if service_name in skip:
        continue
    service = getattr(result.services, service_name)
    if service is not None and service.text:
        print(f"{service_name}:")
        print(f"  {service.text}")
        print()

## Content moderation — nudenet

In [ ]:
from ice9 import CENSOR_LABELS

flagged = result.nsfw_detections(labels=CENSOR_LABELS)

if flagged:
    for detection in flagged:
        print(f"{detection['label']}  confidence={detection['confidence']:.0%}")
else:
    print("No flagged detections.")

## Censoring the image

If nudenet found anything, draw over the flagged regions.

In [ ]:
from IPython.display import display

censored = result.moderation.censor(IMAGE, method="pixelate")
display(censored)

## Colors

In [ ]:
if result.colors is not None:
    print(result.colors.dominant)

## Full result as JSON

In [ ]:
print(result.to_json(indent=2))